# 02 — Phase 2: Human Review UI

Review raw figure crops one patent at a time.  
**Click a thumbnail** to toggle it approved / disapproved.  
Use **Save & Advance** to confirm a patent and move to the next one.  
All decisions are written to `logs/review_meta.json` after every click (crash-safe).

**What gets shown** is controlled by `only_selected` in the cell below — no need to touch `config.yaml`.

### Controls
| Button | Effect |
|---|---|
| ← / → | Navigate between patents |
| Go / Search | Jump to a patent by ID |
| Click thumbnail | Toggle image approved / disapproved |
| Set as Main | Mark that thumbnail as the main image for this patent |
| Disapprove Patent | Reject the entire patent and advance |
| Reapprove Patent | Undo a disapproval or duplicate mark |
| Mark Duplicate | Flag patent as duplicate; enter the source patent ID |
| Needs Split | Flag the active image for sub-crop splitting (Phase 2.5) |
| 1 / 2 architectures | Toggle two-architecture mode for patents with two distinct aircraft |
| Save Arch 1 / 2 | In two-arch mode: save each selection pass separately |
| Save & Advance | Write decisions to review_meta.json and move to the next patent |

In [1]:
import sys; sys.path.insert(0, "..")
import pandas as pd
from pathlib import Path
from src.config_loader import load_config
from src.reviewer import ReviewUI

cfg = load_config()

# All experiment files (review_meta.json, review_export.xlsx) go to the experiment folder
cfg["paths"]["logs"] = cfg["paths"]["experiment"]

# Load the curated patents + their assigned architecture class from selected_patents.csv
CSV_PATH = Path(cfg["paths"]["experiment"]) / "selected_patents.csv"
if not CSV_PATH.exists():
    raise FileNotFoundError(
        f"selected_patents.csv not found at {CSV_PATH}\n"
        "Run 00_patent_filter.ipynb first."
    )
sel = pd.read_csv(CSV_PATH, dtype={"Record Number": str})
patent_ids         = set(sel["Record Number"].str.strip().str.upper())
category_by_patent = dict(zip(sel["Record Number"].str.strip().str.upper(), sel["category"]))
print(f"Loaded {len(patent_ids)} patents from selected_patents.csv")
print(sel["category"].value_counts().to_string())

# The "Architecture:" dropdown on each patent defaults to its assigned class
# (category_by_patent) — change it if the figure doesn't match, "Reset to assigned"
# undoes a correction. Corrections are saved to review_meta.json and override
# selected_patents.csv automatically in 03_processing.
ui = ReviewUI(cfg, patent_ids=patent_ids, category_by_patent=category_by_patent)
ui.show()

Loaded 126 patents from selected_patents.csv
category
compound_helicopter           32
lift_cruise                   32
multirotor                    32
ducted_fan_vectored_thrust    30
Found 125 patents | review_meta.json: /mnt/storage_11tb/Drive_files_to_syncronize/3 - Images DataSets & Labelling Outputs/1639_DS/preliminary analysis v2/review_meta.json


In [2]:
# Run after reviewing to see totals
ui.summary()

Review summary
  Total images : 1928
  Approved     : 1
  Rejected     : 1927
  Needs split  : 0
  Duplicates   : 0 patents


{'total': 1928,
 'approved': 1,
 'rejected': 1927,
 'needs_split': 0,
 'duplicates': 0}

In [3]:
# Export review decisions to Excel (two sheets: Patents + Images)
# Output: Shrouded vs Openrotor/review_export.xlsx
from src.reviewer import export_review_excel
export_review_excel(cfg)

Excel exported → /mnt/storage_11tb/Drive_files_to_syncronize/3 - Images DataSets & Labelling Outputs/1639_DS/preliminary analysis v2/review_export.xlsx
  Patents sheet : 125 rows
  Images sheet  : 1928 rows


PosixPath('/mnt/storage_11tb/Drive_files_to_syncronize/3 - Images DataSets & Labelling Outputs/1639_DS/preliminary analysis v2/review_export.xlsx')